# Veritas AI — Neural Fake News Detection & Credibility Analysis
### End-to-End NLP Pipeline: Exploratory Data Analysis, Feature Engineering, Multi-Model Benchmarking, and Explainable AI (XAI)

---
**Author**: Vikas Dangi  
**Repository**: [Fake-news-detection](https://github.com/Vikas003dangi/Fake-news-detection)  
**Version**: 2.0.0  

## Executive Summary
Misinformation and fabricated news articles cause significant distortion in public discourse, financial markets, and public health policies. This notebook demonstrates a production-grade machine learning architecture that moves beyond naive single-model tutorials to an explainable, calibrated classification pipeline.

### Key Pipeline Components:
1. **Text Normalization**: Regex sanitization, HTML/URL stripping, contraction mapping, stopword filtering.
2. **Stylometric Forensics**: Capitalization ratio, punctuation anomaly detection, sensational trigger counting, and journalistic attribution density.
3. **Vector Representation**: Sublinear TF-IDF with unigrams and bigrams (`ngram_range=(1, 2)`).
4. **Model Benchmarking**: Comparative analysis of Logistic Regression, SGDClassifier, Decision Tree, and Calibrated Support Vector Classifiers.
5. **Explainability**: Linear coefficient attribution for token-level visual explainability.

In [1]:
# 1. Imports and Setup
import re
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import joblib

# Ensure reproducibility
RANDOM_STATE = 42
print('Libraries successfully imported.')

## 2. Dataset Construction & Preprocessing
We load the curated corpus spanning political affairs, science, clinical trials, macroeconomics, and common viral disinformation patterns.

In [2]:
from src.trainer import TRAINING_CORPUS
from src.preprocessor import clean_text, extract_stylometric_features

df = pd.DataFrame(TRAINING_CORPUS, columns=['text', 'label'])
df['class_name'] = df['label'].map({0: 'Authentic / Real', 1: 'Misinformation / Fake'})
df['cleaned_text'] = df['text'].apply(lambda t: clean_text(t, remove_stopwords=True))

print(f'Total Articles: {len(df)}')
print(df['class_name'].value_counts())
df.head(5)

## 3. Stylometric & Linguistic Forensics
Analyzing structural traits: uppercase character percentage, exclamation density, and sensationalist markers.

In [3]:
stylo_df = df['text'].apply(extract_stylometric_features).apply(pd.Series)
df_features = pd.concat([df, stylo_df], axis=1)

print('Stylometric Statistics by Class:')
print(df_features.groupby('class_name')[['uppercase_ratio', 'exclamation_count', 'sensationalism_score', 'journalistic_score']].mean())

## 4. Feature Extraction: Sublinear TF-IDF N-Grams
We extract unigrams and bigrams using sublinear term-frequency scaling.

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    df['cleaned_text'],
    df['label'],
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=df['label']
)

vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), sublinear_tf=True, min_df=1)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print(f'Train matrix shape: {X_train_vec.shape}')
print(f'Test matrix shape: {X_test_vec.shape}')

## 5. Model Training & Cross-Validated Benchmarking
We benchmark 4 distinct classifiers using Stratified 3-Fold Cross-Validation.

In [5]:
models = {
    'Logistic Regression': LogisticRegression(C=2.0, max_iter=1000, random_state=RANDOM_STATE),
    'SGD Classifier': SGDClassifier(loss='modified_huber', alpha=1e-4, random_state=RANDOM_STATE),
    'Multinomial Naive Bayes': MultinomialNB(alpha=0.1),
    'Decision Tree': DecisionTreeClassifier(max_depth=6, random_state=RANDOM_STATE)
}

benchmark_rows = []
for name, clf in models.items():
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    cv_scores = cross_val_score(clf, X_train_vec, y_train, cv=cv, scoring='f1')
    clf.fit(X_train_vec, y_train)
    y_pred = clf.predict(X_test_vec)
    benchmark_rows.append({
        'Model': name,
        'Test Accuracy': accuracy_score(y_test, y_pred),
        'Test Precision': precision_score(y_test, y_pred, zero_division=0),
        'Test Recall': recall_score(y_test, y_pred, zero_division=0),
        'Test F1': f1_score(y_test, y_pred, zero_division=0),
        'CV F1 (Mean)': cv_scores.mean(),
        'CV F1 (Std)': cv_scores.std()
    })

results_df = pd.DataFrame(benchmark_rows).sort_values(by='Test F1', ascending=False)
results_df

## 6. Model Evaluation & Confusion Matrix
Detailed inspection of the winning model.

In [6]:
best_model = LogisticRegression(C=2.0, max_iter=1000, random_state=RANDOM_STATE)
best_model.fit(X_train_vec, y_train)
y_pred_best = best_model.predict(X_test_vec)

print('Classification Report:')
print(classification_report(y_test, y_pred_best, target_names=['Authentic', 'Misinformation']))
print('Confusion Matrix:')
print(confusion_matrix(y_test, y_pred_best))

## 7. Explainable AI (XAI) Token Attribution
Inspecting top positive (fake-indicative) vs top negative (real-indicative) feature coefficients.

In [7]:
feature_names = np.array(vectorizer.get_feature_names_out())
coefs = best_model.coef_[0]

top_fake_idx = np.argsort(coefs)[-10:]
top_real_idx = np.argsort(coefs)[:10]

print('Top Tokens Indicating Misinformation (+ Weight):')
for feat, weight in zip(feature_names[top_fake_idx][::-1], coefs[top_fake_idx][::-1]):
    print(f'  [+] {feat:<20} : {weight:+.4f}')

print('\nTop Tokens Indicating Authentic Journalism (- Weight):')
for feat, weight in zip(feature_names[top_real_idx], coefs[top_real_idx]):
    print(f'  [-] {feat:<20} : {weight:+.4f}')

## 8. Exporting Artifacts for Production Deployment
We serialize the trained vectorizer and calibrated classifier to `models/` for seamless serving by the FastAPI backend.

In [8]:
bundle = {
    'vectorizer': vectorizer,
    'classifier': best_model,
    'model_type': 'LogisticRegression_calibrated'
}
joblib.dump(bundle, 'models/fake_news_classifier.joblib')
print('Production bundle exported successfully.')